In [8]:
import os

os.listdir('//kaggle/input/datasets/selinparmar/cleaned')

['articles_cleaned.parquet',
 'customers_cleaned.parquet',
 'transactions_cleaned.parquet']

In [9]:
customers = pd.read_parquet(
    "//kaggle/input/datasets/selinparmar/cleaned/customers_cleaned.parquet"
)

articles = pd.read_parquet(
    "//kaggle/input/datasets/selinparmar/cleaned/articles_cleaned.parquet"
)

transactions = pd.read_parquet(
    "//kaggle/input/datasets/selinparmar/cleaned/transactions_cleaned.parquet"
)

Cold start recommendation based on Recent + Popular + Diverse + Age-aware

**Merge customer age with transactions**

We need age for age-aware recommendations.

In [10]:
transactions_age = transactions.merge(
    customers[
        ['customer_id', 'age']
    ],
    on='customer_id',
    how='left'
)

transactions_age.head()

,t_dat,customer_id,article_id,price,sales_channel_id,age
0,2018-09-20,000058a12d5b43e67d225668fa1f8d618c13dc232df0ca...,663713001,0.050831,2,24.0
1,2018-09-20,000058a12d5b43e67d225668fa1f8d618c13dc232df0ca...,541518023,0.030492,2,24.0
2,2018-09-20,00007d2de826758b65a93dd24ce629ed66842531df6699...,505221004,0.015237,2,32.0
3,2018-09-20,00007d2de826758b65a93dd24ce629ed66842531df6699...,685687003,0.016932,2,32.0
4,2018-09-20,00007d2de826758b65a93dd24ce629ed66842531df6699...,685687004,0.016932,2,32.0


In [11]:
def age_group(age):
    if age <= 18:
        return "Teen"

    elif age <= 25:
        return "Young Adult"

    elif age <= 35:
        return "Adult"

    elif age <= 50:
        return "Mature"

    else:
        return "Senior"

In [12]:
transactions_age['age_group'] = (
    transactions_age['age']
    .apply(age_group)
)

transactions_age.head()

,t_dat,customer_id,article_id,price,sales_channel_id,age,age_group
0,2018-09-20,000058a12d5b43e67d225668fa1f8d618c13dc232df0ca...,663713001,0.050831,2,24.0,Young Adult
1,2018-09-20,000058a12d5b43e67d225668fa1f8d618c13dc232df0ca...,541518023,0.030492,2,24.0,Young Adult
2,2018-09-20,00007d2de826758b65a93dd24ce629ed66842531df6699...,505221004,0.015237,2,32.0,Adult
3,2018-09-20,00007d2de826758b65a93dd24ce629ed66842531df6699...,685687003,0.016932,2,32.0,Adult
4,2018-09-20,00007d2de826758b65a93dd24ce629ed66842531df6699...,685687004,0.016932,2,32.0,Adult


Use recent purchases only

In [13]:
latest_date = (
    transactions_age['t_dat']
    .max()
)

recent_transactions = (
    transactions_age[
        transactions_age['t_dat']
        >= (
            latest_date -
            pd.Timedelta(days=90)
        )
    ]
)

print(recent_transactions.shape)

(3577236, 7)


Create age-based popularity

In [14]:
age_popularity = (
    recent_transactions
    .groupby(
        ['age_group', 'article_id']
    )
    .size()
    .reset_index(
        name='purchase_count'
    )
)

age_popularity.head()

,age_group,article_id,purchase_count
0,Adult,108775044,16
1,Adult,110065001,1
2,Adult,110065002,1
3,Adult,110065011,1
4,Adult,111565001,106


In [15]:
age_popularity = (
    age_popularity.merge(
        articles,
        on='article_id',
        how='left'
    )
)

age_popularity.head()

,age_group,article_id,purchase_count,product_code,prod_name,product_type_no,product_type_name,product_group_name,graphical_appearance_no,graphical_appearance_name,...,department_name,index_code,index_name,index_group_no,index_group_name,section_no,section_name,garment_group_no,garment_group_name,detail_desc
0,Adult,108775044,16,108775,Strap top,253,Vest top,Garment Upper body,1010016,Solid,...,Jersey Basic,A,Ladieswear,1,Ladieswear,16,Womens Everyday Basics,1002,Jersey Basic,Jersey top with narrow shoulder straps.
1,Adult,110065001,1,110065,OP T-shirt (Idro),306,Bra,Underwear,1010016,Solid,...,Clean Lingerie,B,Lingeries/Tights,1,Ladieswear,61,Womens Lingerie,1017,"Under-, Nightwear","Microfibre T-shirt bra with underwired, moulde..."
2,Adult,110065002,1,110065,OP T-shirt (Idro),306,Bra,Underwear,1010016,Solid,...,Clean Lingerie,B,Lingeries/Tights,1,Ladieswear,61,Womens Lingerie,1017,"Under-, Nightwear","Microfibre T-shirt bra with underwired, moulde..."
3,Adult,110065011,1,110065,OP T-shirt (Idro),306,Bra,Underwear,1010016,Solid,...,Clean Lingerie,B,Lingeries/Tights,1,Ladieswear,61,Womens Lingerie,1017,"Under-, Nightwear","Microfibre T-shirt bra with underwired, moulde..."
4,Adult,111565001,106,111565,20 den 1p Stockings,304,Underwear Tights,Socks & Tights,1010016,Solid,...,Tights basic,B,Lingeries/Tights,1,Ladieswear,62,"Womens Nightwear, Socks & Tigh",1021,Socks and Tights,"Semi shiny nylon stockings with a wide, reinfo..."


In [33]:
global_popular = (
    recent_transactions[
        'article_id'
    ]
    .value_counts()
    .reset_index()
)

global_popular.columns = [
    'article_id',
    'purchase_count'
]

global_popular = (
    global_popular.merge(
        articles,
        on='article_id',
        how='left'
    )
)

In [34]:
def recommend_cold_start(
    age=None,
    top_n=10
):

    # CASE 1: Age available
    if age is not None:

        group = age_group(age)

        recommendations = (
            age_popularity[
                age_popularity['age_group']
                == group
            ]
            .sort_values(
                by='purchase_count',
                ascending=False
            )
        )

    # CASE 2: No age available
    else:

        recommendations = (
            global_popular
            .sort_values(
                by='purchase_count',
                ascending=False
            )
        )

    # Remove unknown products
    recommendations = recommendations[
        recommendations[
            'product_type_name'
        ] != 'Unknown'
    ]

    # Add diversity
    recommendations = (
        recommendations
        .drop_duplicates(
            subset=['product_type_name']
        )
    )

    return (
        recommendations[
            [
                'article_id',
                'product_type_name',
                'product_group_name',
                'colour_group_name',
                'purchase_count'
            ]
        ]
        .head(top_n)
    )

In [35]:
recommend_new_user_by_age(20)

,article_id,product_type_name,product_group_name,colour_group_name,purchase_count
113702,706016001,Trousers,Garment Lower body,Black,1942
107682,372860002,Socks,Socks & Tights,White,1711
117762,759871002,Vest top,Garment Upper body,Black,1489
136427,915526001,Sweater,Garment Upper body,Off White,1466
121974,806388002,T-shirt,Garment Upper body,White,1441
136482,916468003,Cardigan,Garment Upper body,Blue,1384
130219,866383006,Bikini top,Swimwear,Black,1304
136608,918292001,Leggings/Tights,Garment Lower body,Black,1292
127600,850917001,Shirt,Garment Upper body,White,1056
115482,733749001,Top,Garment Upper body,Black,1022


In [36]:
recommend_new_user_by_age(30)

,article_id,product_type_name,product_group_name,colour_group_name,purchase_count
7147,706016001,Trousers,Garment Lower body,Black,1339
25669,866731001,Leggings/Tights,Garment Lower body,Black,1273
3371,610776002,T-shirt,Garment Upper body,Black,1231
25578,866383006,Bikini top,Swimwear,Black,1186
451,372860001,Socks,Socks & Tights,Black,1079
9335,736870001,Vest top,Garment Upper body,Black,867
3090,599580038,Swimwear bottom,Swimwear,Red,854
11448,758034001,Underwear bottom,Underwear,Black,831
18262,817353008,Dress,Garment Full body,Black,763
30809,898694001,Blazer,Garment Upper body,Light Beige,752


In [37]:
recommend_new_user_by_age(45)

,article_id,product_type_name,product_group_name,colour_group_name,purchase_count
44335,751471001,Trousers,Garment Lower body,Black,1017
34284,372860002,Socks,Socks & Tights,White,885
37214,610776002,T-shirt,Garment Upper body,Black,744
39187,678942001,Top,Garment Upper body,Black,662
46132,768912001,Leggings/Tights,Garment Lower body,Black,640
54753,841383002,Vest top,Garment Upper body,Black,638
58182,863595006,Cardigan,Garment Upper body,Black,598
65627,915529003,Sweater,Garment Upper body,Black,582
36941,599580038,Swimwear bottom,Swimwear,Red,573
55931,850917001,Shirt,Garment Upper body,White,571


In [38]:
recommend_new_user_by_age(60)

,article_id,product_type_name,product_group_name,colour_group_name,purchase_count
75822,751471001,Trousers,Garment Lower body,Black,1206
92988,896152002,T-shirt,Garment Upper body,Black,1079
71491,678942001,Top,Garment Upper body,Black,840
87038,856840001,Dress,Garment Full body,Light Beige,822
92995,896169002,Cardigan,Garment Upper body,Grey,777
67323,372860002,Socks,Socks & Tights,White,614
86054,850917001,Shirt,Garment Upper body,White,527
71499,678942054,Sweater,Garment Upper body,Beige,466
91488,884319001,Blouse,Garment Upper body,Beige,405
78841,787946002,Vest top,Garment Upper body,Black,402


In [40]:
recommend_cold_start(age=None)

,article_id,product_type_name,product_group_name,colour_group_name,purchase_count
0,751471001,Trousers,Garment Lower body,Black,4944
2,372860002,Socks,Socks & Tights,White,4297
6,610776002,T-shirt,Garment Upper body,Black,3535
7,918292001,Leggings/Tights,Garment Lower body,Black,3371
9,866383006,Bikini top,Swimwear,Black,3269
13,759871002,Vest top,Garment Upper body,Black,3120
21,850917001,Shirt,Garment Upper body,White,2702
22,915526001,Sweater,Garment Upper body,Off White,2688
24,599580038,Swimwear bottom,Swimwear,Red,2616
26,863595006,Cardigan,Garment Upper body,Black,2586


# New Product Cold Start Strategy

For newly added products with no purchase history:

- Use product metadata
- Recommend based on:
    - product category
    - color similarity
    - garment group
    - textual description similarity

Until interaction data becomes available.

# Day 3 Observations

## Cold Start Problem
Recommendation systems struggle when:
- New users have no purchase history
- New products have no interactions

## Implemented Solution
A hybrid cold-start recommender was developed using:

1. Recent Purchases
- Used last 90 days of data

2. Popularity
- Products ranked by purchase count

3. Diversity
- Removed repetitive categories

4. Age-Aware Personalization
- Recommendations tailored by age group

## Benefits
- Better personalization
- Trending products recommended
- Diverse product categories
- Improved new-user experience

## Conclusion
The recommendation system can now provide intelligent recommendations for cold-start users before sufficient interaction data is available.